In [ ]:
# ==================== PART 1: INSTALL LIBRARIES ====================

import numpy as np
import pandas as pd
import librosa
import torch
import torchaudio
from datasets import load_dataset
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoFeatureExtractor, AutoModel, WhisperProcessor, WhisperForConditionalGeneration
import optuna
import time
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries installed successfully!\n")


In [ ]:
# ==================== PART 2: LOAD DATA & LEAK-FREE SPLIT ====================
import os, sys, json as _json
print("="*60)
print("📥 LOADING DATASET WITH LEAK-FREE SPLIT...")
print("="*60)

# Load dataset
dataset = load_dataset("hustep-lab/ViSEC", trust_remote_code=True)
df = dataset['train'].to_pandas()

print(f"✓ Original dataset size: {len(df)} samples")
print(f"Unique emotions: {df['emotion'].unique()}")
print(f"\nEmotion distribution:")
print(df['emotion'].value_counts())

# Keep only path and emotion
df_clean = df[['path', 'emotion']].copy()

# Encode emotion labels
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['emotion'])
emotion_labels = le.classes_
num_labels = len(emotion_labels)

print(f"\nEmotion mapping:")
for idx, emotion in enumerate(emotion_labels):
    print(f"  {idx}: {emotion}")

# Read fixed split_manifest.json as the single source of truth
manifest_path = os.path.join(os.path.dirname(os.path.abspath(".")), "split_manifest.json")
if not os.path.exists(manifest_path):
    manifest_path = "../split_manifest.json"
if not os.path.exists(manifest_path):
    manifest_path = "split_manifest.json"

print(f"\nLoading manifest from: {manifest_path}")
with open(manifest_path, 'r', encoding='utf-8') as f:
    manifest = _json.load(f)

train_idx = manifest['train_indices']
val_idx = manifest['val_indices']
test_idx = manifest['test_indices']

X_train = df_clean['path'].iloc[train_idx].values
y_train = df_clean['label'].iloc[train_idx].values
X_val = df_clean['path'].iloc[val_idx].values
y_val = df_clean['label'].iloc[val_idx].values
X_test = df_clean['path'].iloc[test_idx].values
y_test = df_clean['label'].iloc[test_idx].values

print(f"\n✓ Train: {len(X_train)} samples")
print(f"✓ Val:   {len(X_val)} samples")
print(f"✓ Test:  {len(X_test)} samples")


In [ ]:
# ==================== PART 3: UTILITY FUNCTIONS ====================

def load_audio(path_dict, sr=16000):
    """Load audio file from bytes or path"""
    try:
        import io
        if isinstance(path_dict, dict) and 'bytes' in path_dict:
            audio_bytes = path_dict['bytes']
            audio, _ = librosa.load(io.BytesIO(audio_bytes), sr=sr)
            return audio
        elif isinstance(path_dict, str):
            audio, _ = librosa.load(path_dict, sr=sr)
            return audio
        elif isinstance(path_dict, dict) and 'path' in path_dict:
            audio, _ = librosa.load(path_dict['path'], sr=sr)
            return audio
    except Exception as e:
        return None
    return None

def plot_confusion_matrix(y_true, y_pred, labels, title):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(title, fontsize=16, pad=20)
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()
    return cm


In [ ]:
# ==================== PART 4: DUAL-STREAM FEATURE EXTRACTION (SEFE + TEFE) ====================
print(f"\n{'='*60}")
print("🎵 EXTRACT DUAL-STREAM FEATURES: WavLM (SEFE) + Whisper (TEFE)")
print(f"{'='*60}")

# Load WavLM model (SEFE stream)
print("Loading WavLM model...")
wavlm_processor = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model = AutoModel.from_pretrained("microsoft/wavlm-base-plus")
wavlm_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
wavlm_model = wavlm_model.to(device)

# Load Whisper model (TEFE stream)
print("Loading Whisper model...")
whisper_processor = WhisperProcessor.from_pretrained("openai/whisper-small")
whisper_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
whisper_model = whisper_model.to(device)
whisper_model.eval()

def extract_acoustic_features(audio, model, processor):
    """Extract acoustic features with WavLM"""
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        features = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return features

def extract_textual_features(audio, model, processor):
    """Extract textual features with Whisper encoder"""
    inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
    inputs = inputs.input_features.to(device)
    with torch.no_grad():
        encoder_outputs = model.get_encoder()(inputs)
        features = encoder_outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
    return features

def extract_dualstream_features(paths, labels, split_name=""):
    """Extract both streams in a single loop — only keeps samples where BOTH succeed."""
    acoustic_list, textual_list, label_list = [], [], []
    for i, (path, label) in enumerate(zip(paths, labels)):
        if i % 200 == 0:
            print(f"  [{split_name}] {i}/{len(paths)}")
        audio = load_audio(path, sr=16000)
        if audio is None:
            continue
        try:
            acoustic = extract_acoustic_features(audio, wavlm_model, wavlm_processor)
            textual = extract_textual_features(audio, whisper_model, whisper_processor)
            acoustic_list.append(acoustic)
            textual_list.append(textual)
            label_list.append(label)
        except Exception:
            continue
    return np.array(acoustic_list), np.array(textual_list), np.array(label_list)

# Extract features for all splits
print("\nExtracting dual-stream features for TRAIN set...")
train_acoustic, train_textual, train_labels_clean = extract_dualstream_features(X_train, y_train, "Train")

print("\nExtracting dual-stream features for VAL set...")
val_acoustic, val_textual, val_labels_clean = extract_dualstream_features(X_val, y_val, "Val")

print("\nExtracting dual-stream features for TEST set...")
test_acoustic, test_textual, test_labels_clean = extract_dualstream_features(X_test, y_test, "Test")

print(f"\n✓ Train: acoustic={train_acoustic.shape}, textual={train_textual.shape}")
print(f"✓ Val:   acoustic={val_acoustic.shape}, textual={val_textual.shape}")
print(f"✓ Test:  acoustic={test_acoustic.shape}, textual={test_textual.shape}")

# Free GPU memory
del wavlm_model, whisper_model
torch.cuda.empty_cache()


In [ ]:
# ==================== PART 5: EARLY FUSION ====================
print(f"\n{'='*60}")
print("🔗 EARLY FUSION: CONCATENATE SEFE + TEFE")
print(f"{'='*60}")

# Concatenate acoustic + textual features
train_fused = np.concatenate([train_acoustic, train_textual], axis=1)
val_fused = np.concatenate([val_acoustic, val_textual], axis=1)
test_fused = np.concatenate([test_acoustic, test_textual], axis=1)

# Normalize features
scaler = StandardScaler()
train_fused = scaler.fit_transform(train_fused)
val_fused = scaler.transform(val_fused)
test_fused = scaler.transform(test_fused)

print(f"✓ Train fused features: {train_fused.shape}")
print(f"✓ Val fused features: {val_fused.shape}")
print(f"✓ Test fused features: {test_fused.shape}")


In [ ]:
# ==================== PART 6: TRAIN CLASSIFIERS (on TRAIN set only) ====================
print(f"\n{'='*60}")
print("🤖 TRAIN 3 CLASSIFIERS")
print(f"{'='*60}")

# 1. Logistic Regression
print("\n1. Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(train_fused, train_labels_clean)
lr_val_pred = lr_model.predict(val_fused)
lr_val_f1 = f1_score(val_labels_clean, lr_val_pred, average='weighted')
print(f"   ✓ Val F1 Score: {lr_val_f1:.4f}")

# 2. Random Forest
print("\n2. Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(train_fused, train_labels_clean)
rf_val_pred = rf_model.predict(val_fused)
rf_val_f1 = f1_score(val_labels_clean, rf_val_pred, average='weighted')
print(f"   ✓ Val F1 Score: {rf_val_f1:.4f}")

# 3. XGBoost with Optuna optimization — tuned on VAL set (not test!)
print("\n3. Training XGBoost with Optuna optimization (on VAL set)...")

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 0.5),
        'random_state': 42
    }
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric='mlogloss')
    model.fit(train_fused, train_labels_clean)
    pred = model.predict(val_fused)
    return f1_score(val_labels_clean, pred, average='weighted')

# Optimize XGBoost — objective uses VAL set
study = optuna.create_study(direction='maximize', study_name='xgboost_optimization')
study.optimize(objective, n_trials=10, show_progress_bar=True)

print(f"\n   ✓ Best Val F1 Score: {study.best_value:.4f}")
print(f"   ✓ Best params: {study.best_params}")

# Train final XGBoost with best params on TRAIN set
xgb_model = XGBClassifier(**study.best_params, use_label_encoder=False, eval_metric='mlogloss')
xgb_model.fit(train_fused, train_labels_clean)
xgb_val_pred = xgb_model.predict(val_fused)
xgb_val_f1 = f1_score(val_labels_clean, xgb_val_pred, average='weighted')
print(f"   ✓ Final Val F1 Score: {xgb_val_f1:.4f}")


In [ ]:
# ==================== PART 7: LATE FUSION — OPTIMIZE ON VAL, EVALUATE ON TEST ====================
print(f"\n{'='*60}")
print("🔀 LATE FUSION: WEIGHTED ENSEMBLE")
print(f"{'='*60}")

# Get probability predictions on VAL set for weight optimization
lr_val_proba = lr_model.predict_proba(val_fused)
rf_val_proba = rf_model.predict_proba(val_fused)
xgb_val_proba = xgb_model.predict_proba(val_fused)

def optimize_ensemble_weights(y_true, pred_proba_lr, pred_proba_rf, pred_proba_xgb):
    """Optimize weights for ensemble using Optuna — on VAL set"""
    def ensemble_objective(trial):
        w1 = trial.suggest_float('w1', 0, 1)
        w2 = trial.suggest_float('w2', 0, 1)
        w3 = trial.suggest_float('w3', 0, 1)
        total = w1 + w2 + w3
        if total == 0:
            return 0.0
        w1, w2, w3 = w1/total, w2/total, w3/total
        ensemble_proba = w1 * pred_proba_xgb + w2 * pred_proba_rf + w3 * pred_proba_lr
        ensemble_pred = np.argmax(ensemble_proba, axis=1)
        f1 = f1_score(y_true, ensemble_pred, average='weighted')
        return f1

    study = optuna.create_study(direction='maximize')
    study.optimize(ensemble_objective, n_trials=100, show_progress_bar=True)
    return study.best_params, study.best_value

# Optimize weights on VAL set (not test!)
print("\nOptimizing ensemble weights on VAL set...")
best_weights, best_ensemble_val_f1 = optimize_ensemble_weights(
    val_labels_clean, lr_val_proba, rf_val_proba, xgb_val_proba
)

w1 = best_weights['w1']
w2 = best_weights['w2']
w3 = best_weights['w3']
total = w1 + w2 + w3
w1, w2, w3 = w1/total, w2/total, w3/total

print(f"\n✓ Optimized weights:")
print(f"   XGBoost: {w1:.4f}")
print(f"   Random Forest: {w2:.4f}")
print(f"   Logistic Regression: {w3:.4f}")
print(f"   Best Val Ensemble F1: {best_ensemble_val_f1:.4f}")

# ==================== FINAL TEST EVALUATION (exactly once) ====================
print(f"\n{'='*60}")
print("📊 FINAL TEST EVALUATION (leak-free)")
print(f"{'='*60}")

# Individual classifier results on TEST
print("\nIndividual classifiers on TEST:")
lr_test_pred = lr_model.predict(test_fused)
rf_test_pred = rf_model.predict(test_fused)
xgb_test_pred = xgb_model.predict(test_fused)

for name, pred in [("LR", lr_test_pred), ("RF", rf_test_pred), ("XGB", xgb_test_pred)]:
    wf1 = f1_score(test_labels_clean, pred, average='weighted')
    acc = accuracy_score(test_labels_clean, pred)
    print(f"  {name}: F1={wf1:.4f}, Acc={acc:.4f}")

# Ensemble on TEST
lr_test_proba = lr_model.predict_proba(test_fused)
rf_test_proba = rf_model.predict_proba(test_fused)
xgb_test_proba = xgb_model.predict_proba(test_fused)

ensemble_proba = w1 * xgb_test_proba + w2 * rf_test_proba + w3 * lr_test_proba
ensemble_pred = np.argmax(ensemble_proba, axis=1)
ensemble_f1 = f1_score(test_labels_clean, ensemble_pred, average='weighted')
ensemble_acc = accuracy_score(test_labels_clean, ensemble_pred)

print(f"\n✓ Ensemble F1 Score (Test): {ensemble_f1:.4f}")
print(f"✓ Ensemble Accuracy (Test): {ensemble_acc:.4f}")

# Summary
results = {
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost (Optimized)', 'Ensemble (Late Fusion)'],
    'F1 Score': [
        f1_score(test_labels_clean, lr_test_pred, average='weighted'),
        f1_score(test_labels_clean, rf_test_pred, average='weighted'),
        f1_score(test_labels_clean, xgb_test_pred, average='weighted'),
        ensemble_f1
    ],
    'Accuracy': [
        accuracy_score(test_labels_clean, lr_test_pred),
        accuracy_score(test_labels_clean, rf_test_pred),
        accuracy_score(test_labels_clean, xgb_test_pred),
        ensemble_acc
    ]
}

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
print("\n" + results_df.to_string(index=False))

print(f"\n{'='*60}")
print("📈 CLASSIFICATION REPORT — ENSEMBLE MODEL")
print(f"{'='*60}")
print(classification_report(test_labels_clean, ensemble_pred, target_names=emotion_labels))

# Confusion Matrix
plot_confusion_matrix(test_labels_clean, ensemble_pred, emotion_labels,
                     "Dual-Stream Ensemble Confusion Matrix")

# Visualize results
plt.figure(figsize=(12, 6))
x = np.arange(len(results_df))

plt.subplot(1, 2, 1)
plt.bar(x, results_df['F1 Score'], color='skyblue', edgecolor='navy')
plt.xlabel('Model', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.title('F1 Score Comparison', fontsize=14)
plt.xticks(x, results_df['Model'], rotation=45, ha='right')
plt.ylim(0, 1)
for i, v in enumerate(results_df['F1 Score']):
    plt.text(i, v + 0.02, f'{v:.4f}', ha='center')

plt.subplot(1, 2, 2)
plt.bar(x, results_df['Accuracy'], color='lightcoral', edgecolor='darkred')
plt.xlabel('Model', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy Comparison', fontsize=14)
plt.xticks(x, results_df['Model'], rotation=45, ha='right')
plt.ylim(0, 1)
for i, v in enumerate(results_df['Accuracy']):
    plt.text(i, v + 0.02, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.show()

print("\n✅ DUAL-STREAM SER SYSTEM COMPLETED!")
